### MINIMIZACAO EICONAL ATE t = 0.1 COM 3 PARAMETROS LIVRES

In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [2]:
# Load experimental data
atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_6961/939602129.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'

In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1):
    return np.exp(-(a1 * q2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1)
    G_minus = G_p(factor, a1)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, m2_func) - T_2(k, q_val, phi, mg, a1, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  



In [ ]:
n_points = 5c 0.1000

def full_int(mg, a1, m2_func, q2_val, sqrt_s):
    # Garante que q_val seja array 1D
    q2_val = np.atleast_1d(q2_val)
    results = []

    for q2 in q2_val:
        def integrand(y, x, mg, a1, m2_func, q2_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            result = k * (
                T_1(k, q2_val, phi, mg, a1, m2_func)
                - T_2(k, q2_val, phi, mg, a1, m2_func)
            ) * jacobian
            return result

        def inner_integral(x):
            integral_real = fixed_quad(
                lambda y: np.real(integrand(y, x, mg, a1, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            integral_imag = fixed_quad(
                lambda y: np.imag(integrand(y, x, mg, a1, m2_func, q2)),
                0, 1, n=n_points
            )[0]
            return integral_real + 1j * integral_imag

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]

In [6]:
lst_chi = []
lst_amp_eik = []
lst_diff_sigma = []

# eps_rel = 1e-4
# eps_abs = 1e-16
# limit = 5000


q_max_chi = 5.0          # limite de q na Eq. 23
b_max     = 15.0



# ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────────
def chi(b_val, mg, a1, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2  # [CORREÇÃO 1] s local, não captura variável global

    def integrand(q_val):
        """Integrando complexo — full_int chamado uma única vez por ponto."""
        # [CORREÇÃO 2] integrando único complexo: evita chamar full_int 2x
        q2_val    = q_val ** 2
        t         = -q2_val
        diff_t    = full_int(mg, a1, m2_func, q2_val, sqrt_s)
        born_amp  = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

    # Integrar partes real e imaginária separadamente (quad requer floats)
    # real_part, _ = quad(
    #     lambda q: np.real(integrand(q)),
    #     0, q_max_chi,
    #     epsrel=eps_rel, epsabs=eps_abs, limit=limit
    # )
    # imag_part, _ = quad(
    #     lambda q: np.imag(integrand(q)),
    #     0, q_max_chi,
    #     epsrel=eps_rel, epsabs=eps_abs, limit=limit
    # )

 
    real_part, _ = fixed_quad(lambda q: np.real(integrand(q)), 0, q_max_chi, n=n_points)
    imag_part, _ = fixed_quad(lambda q: np.imag(integrand(q)), 0, q_max_chi, n=n_points)

    return real_part + 1j * imag_part

In [7]:
def model_function(x, eps, mg, a1, sqrt_s, model_type='log'):

    m2      = m2_log if model_type == 'log' else m2_pl
    s_local = sqrt_s ** 2  # [CORREÇÃO 3] s local, correto para 7/8/13 TeV

    lst_diff_eik = []

    for q2 in x:
        q_val = np.sqrt(q2)  # [CORREÇÃO 6] nome claro: q_val = |q| [GeV]

        def integrand(b_val):
            """Integrando complexo — chi() chamado uma única vez por ponto."""
            # [CORREÇÃO 5] chi() retorna complexo direto; não duplicar a chamada
            chi_val = chi(b_val, mg, a1, eps, m2, sqrt_s)

            return b_val * j0(q_val * b_val) * (1.0 - np.exp(1j* chi_val))

        # real_part, _ = quad(
        #     lambda b: np.real(integrand(b)),
        #     0, b_max,
        #     epsrel=eps_rel, epsabs=eps_abs, limit=limit
        # )
        # imag_part, _ = quad(
        #     lambda b: np.imag(integrand(b)),
        #     0, b_max,
        #     epsrel=eps_rel, epsabs=eps_abs, limit=limit
        # )

        real_part, _ = fixed_quad(lambda b: np.real(integrand(b)), 0, b_max, n=n_points)
        imag_part, _ = fixed_quad(lambda b: np.imag(integrand(b)), 0, b_max, n=n_points)

        amp_eik = 1j * s_local * (real_part + 1j * imag_part)

        # [CORREÇÃO 4] |T|² = Re²+Im² — não descartar a parte real
        diff_sigma_eik = (
            (amp_eik.imag**2 + amp_eik.real**2)
            * (np.pi / s_local**2)
            * 0.389379323          # conversão GeV⁻⁴ → mb
        )
        lst_diff_eik.append(diff_sigma_eik)
    print(f"mg = {mg}, a1  = {a1}, eps = {eps}")

    return np.array(lst_diff_eik)

In [8]:
def model_7(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=7000, model_type='log')

def model_8(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=8000, model_type='log')

def model_13(x, eps, mg, a1):
    return model_function(x, eps, mg, a1, sqrt_s=13000, model_type='log')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7, verbose=True)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8, verbose=True)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13, verbose=True)


chi2_total = chi2_7 + chi2_8 + chi2_13

eps_min = 0.094763
mg_min = 0.93
a1_min = 1.4	

In [9]:

minuit_eik = Minuit(
    chi2_total,
    mg = mg_min,
    a1 = a1_min,
    eps = eps_min
)


minuit_eik.limits['mg'] = (0.3, 0.95)
minuit_eik.limits['a1'] = (0.0001, 5.0)
minuit_eik.limits['eps'] = (0.0001, 0.4)

minuit_eik.print_level = 3

minuit_eik.errordef = 6.251

minuit_eik.simplex()
minuit_eik.simplex()

minuit_eik.migrad(ncall=10000, tol=1e-5, strategy=2)
minuit_eik.migrad(ncall=10000, tol=1e-5, strategy=2)
minuit_eik.migrad(ncall=10000, tol=1e-5, strategy=2)


minuit_eik.hesse()


/home/victorli/miniconda3/envs/two_gluon/lib/python3.11/site-packages/iminuit/minuit.py:139: ErrordefAlreadySetWarning: cost function has an errordef attribute equal to 1.0, you should not override this with Minuit.errordef
  warnings.warn(msg, ErrordefAlreadySetWarning)


mg = 0.93, a1  = 1.4, eps = 0.094763
mg = 0.93, a1  = 1.4, eps = 0.094763
mg = 0.93, a1  = 1.4, eps = 0.094763
(np.float64(0.094763), np.float64(0.93), np.float64(1.4)) -> 1249230.4753821148
D InitialGradientCalculator Calculating initial gradient at point 	[    -0.5545568812      1.218148649    -0.4556307456]	
D InitialGradientCalculator Computed initial gradient for parameter eps value -0.554557 [ -0.0055845 , 0.00556525 ] dirin 0.00557487 grd 2242.56 g2 402263
D InitialGradientCalculator Computed initial gradient for parameter mg value 1.21815 [ -0.0752355 , 0.0953333 ] dirin 0.0852844 grd 146.592 g2 1718.86
D InitialGradientCalculator Computed initial gradient for parameter a1 value -0.455631 [ -0.00624592 , 0.00622686 ] dirin 0.00623639 grd 2004.69 g2 321450
D SimplexBuilder Running with maxfcn 545 minedm 0.6251
mg = 0.93, a1  = 1.4, eps = 0.09571227176333613
mg = 0.93, a1  = 1.4, eps = 0.09571227176333613
mg = 0.93, a1  = 1.4, eps = 0.09571227176333613
(np.float64(0.0957122717633

KeyboardInterrupt: 

In [ ]:
chi2_val = minuit_eik.fval
ndof = chi2_total.ndata - minuit_eik.nfit
chi2_ndof = chi2_val / ndof

print(f"chi2/dof = {chi2_val:.3f} / {ndof} = {chi2_ndof:.3f}")